<a href="https://colab.research.google.com/github/vineet-crypto/Deepfake-Audio-Detection/blob/main/DeepfakeAudioDetector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install librosa torch torchaudio scikit-learn pandas matplotlib seaborn streamlit
import os
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

os.environ['KAGGLE_CONFIG_DIR'] = '/content'
!chmod 600 /content/kaggle.json
!kaggle datasets download -d mohammedabdeldayem/the-fake-or-real-dataset
!unzip -q the-fake-or-real-dataset.zip -d dataset

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 121.4 MB/s eta 0:00:00
chmod: cannot access '/content/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/mohammedabdeldayem/the-fake-or-real-dataset
License(s): GNU Lesser General Public License 3.0
100% 16.0G/16.0G [01:25<00:00, 202MB/s]



In [2]:
import os
import torch
import librosa
import numpy as np
from torch.utils.data import Dataset, DataLoader

# --- Configuration ---
AUDIO_DIR = "/content/dataset/for-norm/for-norm/training/"
TARGET_SR = 16000
MAX_DURATION = 4 # seconds
MEL_BINS = 128

real_audio = os.path.join(AUDIO_DIR, "real")
fake_audio = os.path.join(AUDIO_DIR, "fake")

# --- Feature Extraction Function ---
def compute_melspectrogram(audio_path):
    try:
        waveform, sr = librosa.load(audio_path, sr=TARGET_SR, duration=MAX_DURATION)
        if len(waveform) < TARGET_SR * MAX_DURATION:
            waveform = np.pad(waveform, (0, TARGET_SR * MAX_DURATION - len(waveform)))
        mel_features = librosa.feature.melspectrogram(y=waveform, sr=sr, n_mels=MEL_BINS)
        return librosa.power_to_db(mel_features, ref=np.max)
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None

# --- Custom Dataset Class ---
class DeepfakeAudioDataset(Dataset):
    def __init__(self, real_dir, fake_dir):
        self.audio_filepaths = []
        self.class_targets = []
        self.memory_cache = {}

        for filename in os.listdir(real_dir):
            if filename.endswith(('.wav')):
                self.audio_filepaths.append(os.path.join(real_dir, filename))
                self.class_targets.append(0)

        for filename in os.listdir(fake_dir):
            if filename.endswith(('.wav')):
                self.audio_filepaths.append(os.path.join(fake_dir, filename))
                self.class_targets.append(1)

        if len(self.audio_filepaths) == 0:
            raise ValueError("No audio files found!")

    def __len__(self):
        return len(self.audio_filepaths)

    def __getitem__(self, index):
        if index in self.memory_cache:
            return self.memory_cache[index]

        audio_path = self.audio_filepaths[index]
        extracted_features = compute_melspectrogram(audio_path)

        if extracted_features is None:
            extracted_features = np.zeros((MEL_BINS, int(TARGET_SR * MAX_DURATION / 512)))

        feature_tensor = torch.tensor(extracted_features, dtype=torch.float32).unsqueeze(0)
        target_tensor = torch.tensor(self.class_targets[index], dtype=torch.float32)

        self.memory_cache[index] = (feature_tensor, target_tensor)

        return feature_tensor, target_tensor

entire_audio_dir = DeepfakeAudioDataset(real_audio, fake_audio)
print(f"Successfully loaded {len(entire_audio_dir)} audio files!")

# --- Create an 80/20 train-validation split and initialize batch loaders ---
num_train_samples = int(0.8 * len(entire_audio_dir))
num_val_samples = len(entire_audio_dir) - num_train_samples
training_subset, validation_subset = torch.utils.data.random_split(entire_audio_dir, [num_train_samples, num_val_samples])

training_batch = DataLoader(training_subset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
validation_batch = DataLoader(validation_subset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

Successfully loaded 53868 audio files!


In [3]:
# --- CNN for binary classification of Genuine vs Deepfake audio using 2D Mel Spectrograms ---
class DeepfakeAudioDetectorCNN(nn.Module):
    def __init__(self):
        super(DeepfakeAudioDetectorCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc1 = nn.Linear(64 * 4 * 4, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

# --- Setting up the loss function and Adam optimizer ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepfakeAudioDetectorCNN().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [4]:
# --- Calculates the Equal Error Rate (EER) where the False Acceptance Rate equals the False Rejection Rate ---
def calculate_eer(y_true, y_scores):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    return eer

# --- Runs the complete training cycle and evaluates the model's accuracy, EER, and F1 score ---
def train_evaluate(model, train_loader, val_loader, epochs=5):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(features).squeeze()
            if outputs.ndim == 0: outputs = outputs.unsqueeze(0)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

    # --- Evaluation ---
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for features, labels in val_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features).squeeze()
            if outputs.ndim == 0: outputs = outputs.unsqueeze(0)
            probs = outputs.cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend((probs > 0.5).astype(int))
            all_labels.extend(labels.cpu().numpy())

    # Metrics
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    conf_matrix = confusion_matrix(all_labels, all_preds)
    eer = calculate_eer(all_labels, all_probs)
    tn, fp, fn, tp = conf_matrix.ravel()

    print("\n--- PERFORMANCE REPORT ---")
    print(f"Accuracy: {acc*100:.2f}% | EER: {eer*100:.2f}% | F1: {f1*100:.2f}%")
    print(f"Genuine Acc: {(tn/(tn+fp))*100:.2f}% | Deepfake Acc: {(tp/(tp+fn))*100:.2f}%")
    print("Confusion Matrix:\n", conf_matrix)

    torch.save(model.state_dict(), "deepfake_audio_model.pth")
    print("Model saved as 'deepfake_audio_model.pth'")

train_evaluate(model, training_batch, validation_batch)

Epoch 1/5, Loss: 0.1954
Epoch 2/5, Loss: 0.0265
Epoch 3/5, Loss: 0.0187
Epoch 4/5, Loss: 0.0119
Epoch 5/5, Loss: 0.0113

--- PERFORMANCE REPORT ---
Accuracy: 99.37% | EER: 0.61% | F1: 99.37%
Genuine Acc: 99.01% | Deepfake Acc: 99.72%
Confusion Matrix:
 [[5313   53]
 [  15 5393]]
Model saved as 'deepfake_audio_model.pth'
